# Churn Prediction Challenge

## Machine Learning per l'Analisi Finanziaria — Challenge in-class

Benvenuti alla challenge! Siete un team di consulenti Data Science.
Il vostro cliente (la banca) vi ha fornito un dataset di clienti e vuole
capire **chi rischia di abbandonare** (churn). Avete **60 minuti** per:

1. **Esplorare** i dati
2. **Preprocessare** le feature
3. **Addestrare e confrontare** modelli
4. **Selezionare** il modello migliore con una motivazione
5. **Generare** un report HTML da presentare al cliente

Alla fine, presenterete il vostro lavoro in un **pitch di 3 minuti**
usando il report come supporto visivo.

### Consegna

Scaricate il report HTML generato nell'ultima cella e inviatelo via
mail a **enrico.huber@bip-group.com** con oggetto:
`Challenge ML — SEED XX — Cognomi`

---

## Blocco 0 — Setup e Identificazione

Eseguite questa cella **una sola volta**. Inserite il SEED assegnato
dal docente e i nomi dei componenti del gruppo.

In [ ]:
# =====================================================================
# COMPILATE QUI — il SEED vi è stato assegnato dal docente
# =====================================================================
SEED = 1  # <-- Inserite il vostro SEED (numero da 1 a 15)

COMPONENTI = [
    "Nome Cognome",  # Componente 1
    "Nome Cognome",  # Componente 2
    "Nome Cognome",  # Componente 3
]
# =====================================================================

In [ ]:
# --- Import e setup (NON MODIFICARE) ---
from __future__ import annotations

import io
import json
import sys
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    accuracy_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
    _HAS_XGB = True
except ImportError:
    _HAS_XGB = False
    print("⚠️ XGBoost non disponibile. Potete usare gli altri 3 modelli.")

# Widget (con fallback)
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    _HAS_WIDGETS = True
except ImportError:
    from IPython.display import display, HTML, clear_output
    _HAS_WIDGETS = False
    print("⚠️ ipywidgets non disponibile. Usate la modalità config (celle Python).")

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(SEED)

print(f"✅ Setup completato")
print(f"   SEED: {SEED}")
print(f"   Componenti: {', '.join(COMPONENTI)}")
print(f"   Widget: {'disponibili' if _HAS_WIDGETS else 'NON disponibili (modalità config)'}")
print(f"   XGBoost: {'disponibile' if _HAS_XGB else 'NON disponibile'}")

In [ ]:
# --- Caricamento dataset (NON MODIFICARE) ---

def _find_dataset(seed: int) -> Path:
    """Cerca il dataset nella cartella challenge/datasets/."""
    candidates = [
        Path(f"challenge/datasets/seed_{seed:02d}.csv"),
        Path(f"../challenge/datasets/seed_{seed:02d}.csv"),
        Path(f"datasets/seed_{seed:02d}.csv"),
    ]
    # Cerca anche risalendo dalla cwd
    cwd = Path.cwd()
    for p in [cwd, *cwd.parents]:
        candidate = p / "challenge" / "datasets" / f"seed_{seed:02d}.csv"
        if candidate.exists():
            return candidate
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(
        f"Dataset non trovato per SEED={seed}.\n"
        f"Verifica che il file seed_{seed:02d}.csv esista in challenge/datasets/"
    )

dataset_path = _find_dataset(SEED)
df_raw = pd.read_csv(dataset_path)

print(f"📂 Dataset caricato: {dataset_path}")
print(f"   Righe: {len(df_raw):,}")
print(f"   Colonne: {df_raw.shape[1]}")
print(f"   Churn rate: {df_raw['Exited'].mean():.1%}")
print(f"\n   Colonne: {list(df_raw.columns)}")
display(df_raw.head())

In [ ]:
# --- Storage in-memory per il report (NON MODIFICARE) ---
_report_data = {
    "eda_figures": [],
    "eda_notes": "",
    "preprocessing_config": {},
    "model_results": [],
    "final_model_idx": 0,
    "final_model_figures": [],
    "feature_importance_fig": None,
    "team_notes": {},
    "dataset_info": {
        "n_rows": len(df_raw),
        "n_features": df_raw.shape[1] - 1,
        "churn_rate": float(df_raw["Exited"].mean()),
    },
}
print("✅ Storage report inizializzato")

---
## Blocco 1 — Comprensione dei Dati (EDA)

Selezionate quali analisi volete eseguire. I grafici generati
finiranno automaticamente nel report finale.

In [ ]:
# =====================================================================
# CONFIGURAZIONE EDA — Impostate True/False per ogni analisi
# =====================================================================
EDA_CONFIG = {
    "distribuzione_target": True,
    "statistiche_descrittive": True,
    "distribuzioni_numeriche": True,
    "distribuzioni_categoriche": True,
    "matrice_correlazione": True,
    "boxplot_vs_target": True,
}
# =====================================================================

In [ ]:
# --- Esecuzione EDA (NON MODIFICARE) ---
_report_data["eda_figures"] = []  # reset

numeric_cols = df_raw.select_dtypes(include="number").columns.drop("Exited", errors="ignore")
cat_cols = df_raw.select_dtypes(include="object").columns

if EDA_CONFIG.get("distribuzione_target"):
    fig, ax = plt.subplots(figsize=(6, 4))
    counts = df_raw["Exited"].value_counts()
    ax.bar(["No Churn (0)", "Churn (1)"], counts.values,
           color=["#3498db", "#e74c3c"], edgecolor="black")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 50, f"{v} ({v/len(df_raw):.1%})", ha="center", fontweight="bold")
    ax.set_title("Distribuzione Target (Exited)")
    ax.set_ylabel("Conteggio")
    _report_data["eda_figures"].append(("Distribuzione Target", fig))
    plt.show()

if EDA_CONFIG.get("statistiche_descrittive"):
    print("\n=== Statistiche Descrittive ===")
    display(df_raw.describe())

if EDA_CONFIG.get("distribuzioni_numeriche"):
    n_num = len(numeric_cols)
    ncols = 4
    nrows = (n_num + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3 * nrows))
    axes = axes.flatten() if nrows > 1 else [axes] if ncols == 1 else axes.flatten()
    for i, col in enumerate(numeric_cols):
        axes[i].hist(df_raw[col], bins=30, color="#3498db", edgecolor="black", alpha=0.7)
        axes[i].set_title(col, fontsize=10)
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    fig.suptitle("Distribuzioni Feature Numeriche", fontsize=13, fontweight="bold")
    plt.tight_layout()
    _report_data["eda_figures"].append(("Distribuzioni Numeriche", fig))
    plt.show()

if EDA_CONFIG.get("distribuzioni_categoriche") and len(cat_cols) > 0:
    fig, axes = plt.subplots(1, len(cat_cols), figsize=(5 * len(cat_cols), 4))
    if len(cat_cols) == 1:
        axes = [axes]
    for i, col in enumerate(cat_cols):
        df_raw[col].value_counts().plot.bar(ax=axes[i], color="#2ecc71", edgecolor="black")
        axes[i].set_title(col)
        axes[i].tick_params(axis="x", rotation=45)
    fig.suptitle("Distribuzioni Feature Categoriche", fontsize=13, fontweight="bold")
    plt.tight_layout()
    _report_data["eda_figures"].append(("Distribuzioni Categoriche", fig))
    plt.show()

if EDA_CONFIG.get("matrice_correlazione"):
    fig, ax = plt.subplots(figsize=(10, 8))
    corr = df_raw[numeric_cols.tolist() + ["Exited"]].corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
                ax=ax, square=True, linewidths=0.5)
    ax.set_title("Matrice di Correlazione")
    plt.tight_layout()
    _report_data["eda_figures"].append(("Matrice di Correlazione", fig))
    plt.show()

if EDA_CONFIG.get("boxplot_vs_target"):
    key_features = [c for c in ["Age", "Balance", "CreditScore",
                                "NumOfProducts", "EstimatedSalary"]
                    if c in df_raw.columns]
    if key_features:
        fig, axes = plt.subplots(1, len(key_features),
                                figsize=(5 * len(key_features), 4))
        if len(key_features) == 1:
            axes = [axes]
        for i, col in enumerate(key_features):
            sns.boxplot(data=df_raw, x="Exited", y=col, ax=axes[i],
                        palette=["#3498db", "#e74c3c"])
            axes[i].set_title(f"{col} vs Exited")
        fig.suptitle("Feature vs Target", fontsize=13, fontweight="bold")
        plt.tight_layout()
        _report_data["eda_figures"].append(("Feature vs Target", fig))
        plt.show()

print(f"\n✅ EDA completata — {len(_report_data['eda_figures'])} grafici generati")

### 📝 Osservazioni EDA

Scrivete qui le vostre osservazioni principali sull'analisi esplorativa.
Queste note finiranno nel report finale.

- *Osservazione 1: ...*
- *Osservazione 2: ...*
- *Osservazione 3: ...*

In [ ]:
# =====================================================================
# COMPILATE QUI — Le vostre osservazioni sull'EDA
# =====================================================================
EDA_NOTES = """
Scrivete qui le vostre osservazioni principali sull'EDA.
Queste note appariranno nel report finale.

Esempio:
- Il churn rate è del X%, il dataset è sbilanciato/bilanciato
- La feature Age mostra una distribuzione ...
- La correlazione tra X e Y è ...
"""

_report_data["eda_notes"] = EDA_NOTES
print("✅ Note EDA salvate")

---
## Blocco 2 — Preprocessing

Configurate le vostre scelte di preprocessing. Poi eseguite la cella
successiva per applicarle.

In [ ]:
# =====================================================================
# CONFIGURAZIONE PREPROCESSING — Modificate le vostre scelte
# =====================================================================
PREPROCESS_CONFIG = {
    # Scaling: "standard" (StandardScaler), "minmax" (MinMaxScaler), "none"
    "scaling": "standard",

    # Feature engineering: creare la feature balance_is_zero?
    "balance_is_zero": True,

    # Feature da ESCLUDERE dal modello (lista di nomi colonna)
    # Suggerimento: alcune colonne non sono utili per la predizione
    "exclude_features": [],

    # Percentuale test split (0.15, 0.20, 0.25, 0.30)
    "test_size": 0.20,
}
# =====================================================================

In [ ]:
# --- Applicazione preprocessing (NON MODIFICARE) ---

df = df_raw.copy()

# Feature engineering
if PREPROCESS_CONFIG.get("balance_is_zero"):
    df["balance_is_zero"] = (df["Balance"] == 0).astype(int)
    print("✅ Feature 'balance_is_zero' creata")

# Separazione target
y = df["Exited"]
X = df.drop(columns=["Exited"])

# Esclusione feature
exclude = PREPROCESS_CONFIG.get("exclude_features", [])
if exclude:
    X = X.drop(columns=[c for c in exclude if c in X.columns], errors="ignore")
    print(f"✅ Feature escluse: {exclude}")

# Encoding categoriche
cat_cols_to_encode = X.select_dtypes(include="object").columns.tolist()
label_encoders = {}
for col in cat_cols_to_encode:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le
if cat_cols_to_encode:
    print(f"✅ Encoding categoriche: {cat_cols_to_encode}")

# Split
test_size = PREPROCESS_CONFIG.get("test_size", 0.20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=test_size, stratify=y, random_state=SEED
)
print(f"✅ Split: train={len(X_train)}, val={len(X_val)} "
      f"(test_size={test_size:.0%})")
print(f"   Churn rate — train: {y_train.mean():.1%}, val: {y_val.mean():.1%}")

# Scaling
scaling = PREPROCESS_CONFIG.get("scaling", "standard")
if scaling == "standard":
    scaler = StandardScaler()
elif scaling == "minmax":
    scaler = MinMaxScaler()
else:
    scaler = None

if scaler is not None:
    X_train_s = pd.DataFrame(scaler.fit_transform(X_train),
                             columns=X_train.columns, index=X_train.index)
    X_val_s = pd.DataFrame(scaler.transform(X_val),
                           columns=X_val.columns, index=X_val.index)
    print(f"✅ Scaling: {scaling}")
else:
    X_train_s = X_train.copy()
    X_val_s = X_val.copy()
    print("✅ Scaling: nessuno")

# Salva config nel report
_report_data["preprocessing_config"] = {
    "Scaling": scaling,
    "balance_is_zero": PREPROCESS_CONFIG.get("balance_is_zero"),
    "Feature escluse": exclude if exclude else "Nessuna",
    "Test size": f"{test_size:.0%}",
    "Feature finali": len(X_train_s.columns),
    "Righe train": len(X_train_s),
    "Righe val": len(X_val_s),
}

print(f"\n📊 Feature finali ({len(X_train_s.columns)}): {list(X_train_s.columns)}")

---
## Blocco 3 — Modellazione

Potete **eseguire questa cella più volte** con modelli e iperparametri
diversi. Ogni esecuzione aggiunge una riga alla tabella comparativa.

Modelli disponibili: `LogisticRegression`, `DecisionTree`,
`RandomForest`, `XGBoost`

In [ ]:
# =====================================================================
# CONFIGURAZIONE MODELLO — Modificate e ri-eseguite questa cella
# per provare combinazioni diverse
# =====================================================================

# Scegliete il modello: "LogisticRegression", "DecisionTree",
#                       "RandomForest", "XGBoost"
MODEL_NAME = "RandomForest"

# Iperparametri (modificate quelli del modello scelto)
MODEL_PARAMS = {
    # --- LogisticRegression ---
    # "C": 1.0,              # Regolarizzazione (0.01 - 10.0)
    # "penalty": "l2",       # "l1" o "l2"
    # "class_weight": "balanced",  # "balanced" o None

    # --- DecisionTree ---
    # "max_depth": 6,        # Profondità (2 - 15)
    # "min_samples_leaf": 5, # Min campioni per foglia (1 - 50)
    # "class_weight": "balanced",

    # --- RandomForest ---
    "n_estimators": 200,     # Numero alberi (50 - 300)
    "max_depth": 10,         # Profondità (3 - 20)
    "min_samples_leaf": 5,   # Min campioni per foglia (1 - 50)
    "class_weight": "balanced",

    # --- XGBoost ---
    # "n_estimators": 200,   # Numero alberi (50 - 300)
    # "max_depth": 5,        # Profondità (3 - 8)
    # "learning_rate": 0.1,  # Learning rate (0.01 - 0.3)
}
# =====================================================================

In [ ]:
# --- Addestramento e valutazione (NON MODIFICARE) ---

def _build_model(name: str, params: dict, seed: int, y_train=y_train):
    """Costruisce il modello con i parametri specificati."""
    p = params.copy()
    if name == "LogisticRegression":
        solver = "liblinear" if p.get("penalty") == "l1" else "lbfgs"
        return LogisticRegression(
            C=p.get("C", 1.0),
            penalty=p.get("penalty", "l2"),
            class_weight=p.get("class_weight", "balanced"),
            max_iter=1000,
            solver=solver,
            random_state=seed,
        )
    elif name == "DecisionTree":
        return DecisionTreeClassifier(
            max_depth=p.get("max_depth", 6),
            min_samples_leaf=p.get("min_samples_leaf", 5),
            class_weight=p.get("class_weight", "balanced"),
            random_state=seed,
        )
    elif name == "RandomForest":
        return RandomForestClassifier(
            n_estimators=p.get("n_estimators", 200),
            max_depth=p.get("max_depth", 10),
            min_samples_leaf=p.get("min_samples_leaf", 5),
            class_weight=p.get("class_weight", "balanced"),
            random_state=seed,
            n_jobs=-1,
        )
    elif name == "XGBoost":
        if not _HAS_XGB:
            raise ImportError("XGBoost non installato")
        n_pos = y_train.sum()
        n_neg = len(y_train) - n_pos
        spw = n_neg / n_pos if n_pos > 0 else 1.0
        return XGBClassifier(
            n_estimators=p.get("n_estimators", 200),
            max_depth=p.get("max_depth", 5),
            learning_rate=p.get("learning_rate", 0.1),
            scale_pos_weight=spw,
            random_state=seed,
            eval_metric="auc",
            use_label_encoder=False,
            n_jobs=-1,
        )
    else:
        raise ValueError(f"Modello '{name}' non supportato")


# Addestramento
print(f"🔄 Addestramento {MODEL_NAME}...")
model = _build_model(MODEL_NAME, MODEL_PARAMS, SEED)
model.fit(X_train_s, y_train)

# Predizioni
y_prob_val = model.predict_proba(X_val_s)[:, 1]
y_pred_val = (y_prob_val >= 0.5).astype(int)

# Metriche
auc = roc_auc_score(y_val, y_prob_val)
f1 = f1_score(y_val, y_pred_val)
prec = precision_score(y_val, y_pred_val)
rec = recall_score(y_val, y_pred_val)
acc = accuracy_score(y_val, y_pred_val)

result = {
    "model_name": MODEL_NAME,
    "params": MODEL_PARAMS.copy(),
    "auc": auc,
    "f1": f1,
    "precision": prec,
    "recall": rec,
    "accuracy": acc,
    "_model_obj": model,  # salva per dopo
}
_report_data["model_results"].append(result)

# Stampa risultati
run_n = len(_report_data["model_results"])
print(f"\n✅ Run #{run_n} — {MODEL_NAME}")
print(f"   ROC-AUC:   {auc:.4f}")
print(f"   F1-Score:  {f1:.4f}")
print(f"   Precision: {prec:.4f}")
print(f"   Recall:    {rec:.4f}")
print(f"   Accuracy:  {acc:.4f}")

# Confusion Matrix
fig_cm, ax_cm = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_val, y_pred_val, display_labels=["No Churn", "Churn"],
    cmap="Blues", ax=ax_cm)
ax_cm.set_title(f"Run #{run_n} — {MODEL_NAME} (AUC={auc:.4f})")
plt.tight_layout()
plt.show()

# Tabella cumulativa
print(f"\n{'='*70}")
print(f"{'#':>3s}  {'Modello':<20s}  {'AUC':>8s}  {'F1':>8s}  {'Prec':>8s}  {'Rec':>8s}")
print(f"{'─'*70}")
for i, r in enumerate(_report_data["model_results"]):
    marker = " ← best" if r["auc"] == max(rr["auc"] for rr in _report_data["model_results"]) else ""
    print(f"{i+1:>3d}  {r['model_name']:<20s}  {r['auc']:8.4f}  {r['f1']:8.4f}  "
          f"{r['precision']:8.4f}  {r['recall']:8.4f}{marker}")
print(f"{'='*70}")
print(f"\n💡 Per provare un altro modello, modificate MODEL_NAME e MODEL_PARAMS")
print(f"   nella cella sopra e ri-eseguite entrambe le celle.")

---
## Blocco 4 — Selezione del Modello Finale

Dopo aver provato diversi modelli, selezionate quello che volete
presentare al cliente.

In [ ]:
# =====================================================================
# SELEZIONE — Inserite il numero della run scelta (dalla tabella sopra)
# =====================================================================
SELECTED_RUN = 1  # <-- Numero della run da selezionare (1, 2, 3, ...)
# =====================================================================

In [ ]:
# --- Analisi modello finale (NON MODIFICARE) ---

idx = SELECTED_RUN - 1
if idx < 0 or idx >= len(_report_data["model_results"]):
    print(f"❌ Run #{SELECTED_RUN} non valida. Avete {len(_report_data['model_results'])} run.")
else:
    _report_data["final_model_idx"] = idx
    final = _report_data["model_results"][idx]
    final_model = final["_model_obj"]
    print(f"✅ Modello finale selezionato: Run #{SELECTED_RUN} — {final['model_name']}")
    print(f"   AUC={final['auc']:.4f}, F1={final['f1']:.4f}")

    # ROC curve
    y_prob_final = final_model.predict_proba(X_val_s)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_prob_final)
    fig_roc, ax_roc = plt.subplots(figsize=(6, 6))
    ax_roc.plot(fpr, tpr, lw=2, color="#3498db",
                label=f"{final['model_name']} (AUC={final['auc']:.4f})")
    ax_roc.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Random")
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate")
    ax_roc.set_title("ROC Curve — Modello Finale")
    ax_roc.legend()
    ax_roc.set_aspect("equal")
    plt.tight_layout()

    # Confusion Matrix
    y_pred_final = (y_prob_final >= 0.5).astype(int)
    fig_cm2, ax_cm2 = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_val, y_pred_final, display_labels=["No Churn", "Churn"],
        cmap="Blues", ax=ax_cm2)
    ax_cm2.set_title(f"Confusion Matrix — {final['model_name']}")
    plt.tight_layout()

    _report_data["final_model_figures"] = [
        ("ROC Curve", fig_roc),
        ("Confusion Matrix", fig_cm2),
    ]
    plt.show()

    # Feature importance
    if hasattr(final_model, "feature_importances_"):
        imp = pd.Series(final_model.feature_importances_,
                        index=X_train_s.columns).sort_values(ascending=True)
        fig_fi, ax_fi = plt.subplots(figsize=(8, max(4, len(imp) * 0.35)))
        imp.plot.barh(ax=ax_fi, color="#3498db", edgecolor="black")
        ax_fi.set_title(f"Feature Importance — {final['model_name']}")
        ax_fi.set_xlabel("Importanza")
        plt.tight_layout()
        _report_data["feature_importance_fig"] = fig_fi
        plt.show()
    elif hasattr(final_model, "coef_"):
        coef = pd.Series(final_model.coef_[0],
                         index=X_train_s.columns).sort_values(ascending=True)
        fig_fi, ax_fi = plt.subplots(figsize=(8, max(4, len(coef) * 0.35)))
        colors = ["#e74c3c" if v < 0 else "#3498db" for v in coef]
        coef.plot.barh(ax=ax_fi, color=colors, edgecolor="black")
        ax_fi.set_title(f"Coefficienti — {final['model_name']}")
        ax_fi.set_xlabel("Coefficiente")
        plt.tight_layout()
        _report_data["feature_importance_fig"] = fig_fi
        plt.show()
    else:
        _report_data["feature_importance_fig"] = None

    # Classification report
    print(f"\n=== Classification Report ===")
    print(classification_report(y_val, y_pred_final,
                                target_names=["No Churn", "Churn"]))

### 💾 Esportazione Modello\n\nEseguite questa cella per salvare il modello serializzato.\nDovete allegarlo insieme all'HTML nell'email di consegna."

In [ ]:
# --- Salvataggio modello (NON MODIFICARE) ---
import joblib

idx = _report_data["final_model_idx"]
if not _report_data["model_results"] or idx >= len(_report_data["model_results"]):
    print("❌ Selezionate prima il modello finale nel Blocco 4.")
else:
    final = _report_data["model_results"][idx]
    final_model = final["_model_obj"]

    # Bundle con tutto il necessario per riprodurre le predizioni
    bundle = {
        "seed": SEED,
        "componenti": COMPONENTI,
        "model_name": final["model_name"],
        "model": final_model,
        "scaler": scaler,
        "label_encoders": label_encoders,
        "feature_order": list(X_train_s.columns),
        "exclude_features": PREPROCESS_CONFIG.get("exclude_features", []),
        "balance_is_zero": PREPROCESS_CONFIG.get("balance_is_zero", True),
        "train_metrics": {
            "auc": final["auc"],
            "f1": final["f1"],
            "precision": final["precision"],
            "recall": final["recall"],
            "accuracy": final["accuracy"],
        },
    }

    # Salva in challenge/outputs/
    _, out_dir = _find_report_generator()
    model_path = Path(out_dir) / f"model_seed_{SEED:02d}.joblib"
    joblib.dump(bundle, model_path)

    print(f"✅ Modello salvato: {model_path}")
    print(f"   Modello: {final['model_name']}")
    print(f"   AUC train: {final['auc']:.4f}")
    print(f"\n📧 Allegate questo file insieme all'HTML nell'email di consegna.")

### 📝 Interpretazione e Raccomandazioni

Compilate le celle seguenti con le vostre riflessioni.
Questi testi finiscono nel report e guideranno il vostro pitch.

In [ ]:
# =====================================================================
# COMPILATE QUI — Le vostre riflessioni
# =====================================================================

TEAM_NOTES = {
    "motivazione": """
Perché avete scelto questo modello? Quali criteri avete usato?
(Sostituite questo testo con la vostra risposta)
""",

    "business_insight": """
Quali insight di business emergono dalla vostra analisi?
Quali pattern avete osservato nei dati?
(Sostituite questo testo con la vostra risposta)
""",

    "strategia_retention": """
Quale strategia di retention proporreste alla banca
basandovi sui risultati del vostro modello?
(Sostituite questo testo con la vostra risposta)
""",
}

_report_data["team_notes"] = TEAM_NOTES
print("✅ Note del team salvate")
# =====================================================================

---\n## Blocco 5 — Generazione Report\n\nEseguite questa cella per generare il report HTML finale.\n\nDopo l'esecuzione, scaricate **entrambi i file** dal file explorer\n(click destro → Download) e inviateli a **enrico.huber@bip-group.com**:\n\n- `report_seed_XX.html` — il report con grafici e analisi\n- `model_seed_XX.joblib` — il modello serializzato per la valutazione

In [ ]:
# --- Generazione report HTML (NON MODIFICARE) ---

# Importa il generatore
def _find_report_generator():
    """Trova e importa il modulo report_generator."""
    cwd = Path.cwd()
    for p in [cwd, *cwd.parents]:
        candidate = p / "challenge" / "report_generator.py"
        if candidate.exists():
            sys.path.insert(0, str(candidate.parent))
            import report_generator
            return report_generator, candidate.parent / "outputs"
    # Fallback: stessa directory
    try:
        import report_generator
        return report_generator, Path("outputs")
    except ImportError:
        raise ImportError(
            "report_generator.py non trovato. "
            "Assicuratevi che sia nella cartella challenge/."
        )

rg, output_dir = _find_report_generator()

# Prepara dati per il report (rimuovi oggetti modello non serializzabili)
clean_results = []
for r in _report_data["model_results"]:
    clean = {k: v for k, v in r.items() if k != "_model_obj"}
    clean_results.append(clean)

report_path = rg.generate_html_report(
    seed=SEED,
    componenti=COMPONENTI,
    dataset_info=_report_data["dataset_info"],
    eda_figures=_report_data["eda_figures"],
    eda_notes=_report_data["eda_notes"],
    preprocessing_config=_report_data["preprocessing_config"],
    model_results=clean_results,
    final_model_idx=_report_data["final_model_idx"],
    final_model_figures=_report_data["final_model_figures"],
    feature_importance_fig=_report_data["feature_importance_fig"],
    team_notes=_report_data["team_notes"],
    output_dir=output_dir,
)

model_path = Path(output_dir) / f"model_seed_{SEED:02d}.joblib"
cognomi = ', '.join(c.split()[-1] for c in COMPONENTI if c.strip())

print(f"\n{'='*60}")
print(f"  ✅ FILE PRONTI PER LA CONSEGNA")
print(f"  📄 Report:  {report_path}")
print(f"  🤖 Modello: {model_path}")
print(f"{'='*60}")
print(f"\n📧 Scaricate ENTRAMBI i file dal file explorer di VS Code")
print(f"   (click destro sul file → Download)")
print(f"   e inviateli a: enrico.huber@bip-group.com")
print(f"   Oggetto: Challenge ML — SEED {SEED:02d} — {cognomi}")
if not model_path.exists():
    print(f"\n⚠️  Il modello non è ancora stato esportato.")
    print(f"   Eseguite prima la cella '💾 Esportazione Modello' nel Blocco 4.")